# Fixed plot size and aspect ratio — `nb.set_plot_size()`

`figsize` sizes the **whole figure**. Everything else — the suptitle, the legend,
axis labels, a colorbar, and above all *how many subplots the call produced* —
then eats into that figure, so the actual drawing area moves around from plot to
plot. Two charts in the same notebook end up different shapes, which makes them
hard to compare and worse to paste side by side into a report.

`set_plot_size` pins the **plot area** instead, and by default it pins **one
subplot panel**:

```python
nb.set_plot_size(4, 3)   # every panel is 4x3 inches, in every plot, always
```

The figure is then grown to `plot area + margins`, so the title, legend and
labels are absorbed by the margins rather than taken out of the data.

This notebook shows the problem, the fix, that the fix holds up against every
decoration unichart can add, and the escape hatch for the old behaviour.

## Setup

In [ ]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
from unichart import UnichartNotebook


def sample_runs(n_runs=3, n=60, seed=11):
    """A few 'engine runs': an RPM sweep with several sensor channels each."""
    rng = np.random.default_rng(seed)
    frames = []
    for i in range(n_runs):
        rpm = np.linspace(1200, 2700, n)
        frames.append(pd.DataFrame({
            "RPM":      rpm,
            "Temp":     180 + 0.05 * rpm + rng.normal(0, 4, n) + 12 * i,
            "Pressure": 14 + 0.004 * rpm + rng.normal(0, 0.4, n) - 0.6 * i,
            "Fuel":     8 + 0.012 * rpm + rng.normal(0, 0.6, n),
            "Vibration": 0.4 + 0.0004 * rpm ** 1.05 / 30 + rng.normal(0, 0.05, n),
            "Load":     np.linspace(20, 95, n),
        }))
    return frames


RUN_TITLES = ["Run A — hot day", "Run B — standard", "Run C — cold day"]

# Several cells below call plot() in a loop purely to measure the result, and
# each of those calls would otherwise leave a stray "⧉ copy" button behind with
# no plot under it. Turning the buttons off keeps the output readable; drop this
# line in your own notebooks.
_COPY_BUTTONS = False

### A helper that reads the panel size back off the figure

Rather than eyeballing it, we measure. The plot area is the figure minus its
margins; one panel is that area times the subplot's share of it, which is
exactly what the axis `domain` records. (Overlaying axes — the extra Y axes of
`plot_ymult` — are skipped, since they sit on the same panel.)

In [ ]:
def panel_size(fig):
    """Size of a single subplot panel, in inches."""
    m = fig.layout.margin
    mv = lambda v, d: d if v is None else v
    inner_w = fig.layout.width - mv(m.l, 80) - mv(m.r, 80)
    inner_h = fig.layout.height - mv(m.t, 100) - mv(m.b, 80)
    xs = [ax.domain for ax in fig.select_xaxes() if not ax.overlaying and ax.domain]
    ys = [ax.domain for ax in fig.select_yaxes() if not ax.overlaying and ax.domain]
    fx = max(d[1] - d[0] for d in xs) if xs else 1.0
    fy = max(d[1] - d[0] for d in ys) if ys else 1.0
    return inner_w * fx / 100, inner_h * fy / 100


def report(nb, label):
    """Print one row: panel size, aspect ratio and overall figure size."""
    w, h = panel_size(nb.last_fig)
    print(f"{label:<26s} panel {w:5.2f} x {h:5.2f} in   aspect {w / h:5.2f}"
          f"   figure {nb.last_fig.layout.width / 100:5.2f} x "
          f"{nb.last_fig.layout.height / 100:5.2f} in")

## 1. The problem — panels change shape as you add variables

`nb.plot()` puts one subplot per Y variable. With only `figsize` in play the
figure stays the same size, so those panels are carved out of a fixed area:
plot more variables and each panel gets smaller *and a different shape*.

Note the aspect column below — it swings from tall-and-narrow to short-and-wide
across four calls that a reader would expect to look like a set.

In [ ]:
nb = UnichartNotebook()
nb.set_copy_buttons(_COPY_BUTTONS)
for frame, title in zip(sample_runs(), RUN_TITLES):
    nb.load_df(frame, title=title)

PANEL_COUNTS = (["Temp"],
                ["Temp", "Pressure"],
                ["Temp", "Pressure", "Fuel"],
                ["Temp", "Pressure", "Fuel", "Vibration", "Load"])

for y in PANEL_COUNTS:
    nb.plot(x="RPM", y=y)
    report(nb, f"figsize only, {len(y)} var(s)")

The panels are never the same twice. Now look at one of them.

In [ ]:
nb.plot(x="RPM", y=["Temp", "Pressure"])

## 2. The fix — pin the panel

One call, and it sticks for every plot afterwards.

In [ ]:
nb.set_plot_size(4, 3)          # inches, per subplot panel

for y in PANEL_COUNTS:
    nb.plot(x="RPM", y=y)
    report(nb, f"pinned 4x3, {len(y)} var(s)")

Every panel is exactly 4 x 3 inches. The *figure* is what changes now — it grows
to fit the grid — which is the trade that makes the panels comparable.

In [ ]:
nb.plot(x="RPM", y=["Temp", "Pressure"])

In [ ]:
nb.plot(x="RPM", y=["Temp", "Pressure", "Fuel", "Vibration", "Load"])

## 3. The pin survives everything that normally moves the plot area

Each of these adds something that would otherwise take space out of the data
region — extra title lines, a footer, a legend moved to the side, a legend tall
enough to wrap, a colorbar, multiple Y axes. The panel stays 4 x 3.

In [ ]:
nb.plot(x="RPM", y="Temp")
report(nb, "plain")

nb.plot(x="RPM", y="Temp", suptitle="Engine test campaign — hot / standard / "
                                    "cold day comparison, revision 4")
report(nb, "long suptitle")

nb.plot(x="RPM", y="Temp", footer="source: rig A\nreduced 2026-08-30")
report(nb, "multi-line footer")

nb.plot(x="RPM", y="Temp", legend="right")
report(nb, "legend on the right")

nb.plot(x="RPM", y="Temp", legend="off")
report(nb, "no legend")

nb.hue("all", "Load")                      # colorbar on the right
nb.plot(x="RPM", y="Temp")
report(nb, "hue colorbar")
nb.hue("all", None)

nb.contour(x="RPM", y="Temp", z="Fuel")
report(nb, "contour + colorbar")

nb.bar(x="RPM", y="Temp")
report(nb, "bar")

nb.plot_ymult(x="RPM", y=["Temp", "Pressure", "Fuel"])
report(nb, "plot_ymult, 3 axes")

`plot_ymult` is the interesting one. It stacks a grouped legend that can run
several rows tall, and it hangs extra Y axes off the right-hand side — each of
which keeps a fixed pixel slot for its tick labels and title, so narrowing the
plot pushes the axes further out rather than crushing them into each other.

In [ ]:
nb.plot_ymult(x="RPM", y=["Temp", "Pressure", "Fuel"],
              suptitle="Three channels, one panel — still 4 x 3")

Add two more channels and the extra axes step outward, each keeping its own
labels legible; the data region is still 4 x 3.

In [ ]:
nb.plot_ymult(x="RPM", y=["Temp", "Pressure", "Fuel", "Vibration", "Load"],
              suptitle="Five channels — the axes spread, the panel does not")
report(nb, "plot_ymult, 5 axes")

### A wrapping legend

Load a dozen datasets and the above-legend wraps to several rows. That pushes
the *figure* taller; it does not shrink the panel.

In [ ]:
wide = UnichartNotebook()
wide.set_copy_buttons(_COPY_BUTTONS)
for i, frame in enumerate(sample_runs(n_runs=12, seed=3)):
    wide.load_df(frame, title=f"Run {chr(65 + i)} — configuration {i + 1}")
wide.set_plot_size(4, 3)
wide.plot(x="RPM", y="Temp")
report(wide, "12 datasets, wrapped legend")
wide.plot(x="RPM", y="Temp")

## 4. Pin one dimension only

Pass just the dimension you care about; the other keeps following `figsize`.
Pinning only the height is a good way to get a row of plots that line up
vertically while their widths follow the content.

In [ ]:
nb.set_plot_size(height=3)              # width still comes from figsize
for y in PANEL_COUNTS:
    nb.plot(x="RPM", y=y)
    report(nb, f"height pinned, {len(y)} var(s)")

Each call **replaces** the previous setting rather than merging with it, so
`set_plot_size(height=3)` above dropped the earlier width pin. Pass both
dimensions whenever you want both held.

## 5. `per_subplot=False` — pin the whole grid instead

The default sizes one panel. `per_subplot=False` pins the *combined* grid area,
which is what `set_plot_size` did before per-panel sizing: the figure stays put
and the panels shrink as the grid grows. Useful when the constraint is the space
the figure has to fit into — a fixed slide or column width — rather than the
shape of the panels.

`per_subplot` is part of the setting a call replaces, so pass it every time you
want it.

In [ ]:
nb.set_plot_size(9, 6, per_subplot=False)   # the whole grid is 9x6 in
for y in PANEL_COUNTS:
    nb.plot(x="RPM", y=y)
    report(nb, f"grid pinned 9x6, {len(y)} var(s)")

The figure holds at a constant size across all three, and the panels shrink to
share it — the opposite trade from the default.

## 6. Clearing the pin

`reset=True`, no arguments at all, or the `reset_format` scope — all equivalent.

In [ ]:
nb.set_plot_size(reset=True)
print("plot_size:", nb.plot_size)

nb.set_plot_size(4, 3, per_subplot=False)
nb.reset_format("plot_size")
print("plot_size:", nb.plot_size, " per_subplot:", nb.plot_size_per_subplot)

nb.plot(x="RPM", y=["Temp", "Pressure"])
report(nb, "unpinned again")

## 7. Putting it to work

The usual pattern is one line near the top of the notebook, after which every
figure in the document is built from panels of the same size and shape:

```python
nb = UnichartNotebook()
nb.load(...)
nb.set_plot_size(4, 3)      # every panel, every plot, from here on
```

Worth knowing:

- **The figure grows with the grid.** Five 6-inch panels side by side is a
  ~22-inch figure. Use a smaller per-panel size, or `ncols=1`, if that is
  awkward in a notebook cell.
- **`figsize` still drives any unpinned dimension**, and still sets the figure
  size when nothing is pinned at all.
- **Very long tick labels** can still expand a margin and leave that dimension
  slightly short. Widen `figsize` or shorten the labels if you hit it.
- **Dashboards are unaffected** — each board panel is drawn at the size the
  board gives it (`width` / `height` on `dashboard()`), so the pin governs
  notebook figures, not board tiles.
- `nb.help('set_plot_size')` has the full reference.